# Finding depth and breadth of conditionality in SPDX

### 1. Setting up

Conditions: Manually extracted list of all conditions in SPDX 3.0

Schema: My manually extracted schema from the SPDX specifications. I chose the manually extracted schema over the official schema at this stage in order to be consistent with my other work, which had findings based off of my own schema.

In [24]:
import json

CLASSES_FILE = 'classes_core.txt'
SCHEMA_FILE = 'schema.json'
OUTPUT_FILE = "classes_breadth.csv"


### 2. Loading classes and schema

In [ ]:
with open(CLASSES_FILE, 'r') as f:
    classes = {line.strip() for line in f}

print(f"{len(classes)}  classes loaded: {classes}\n")

with open(SCHEMA_FILE, 'r') as f:
    schema = json.load(f)

20  classes loaded: {'NamespaceMap', 'Artifact', 'DictionaryEntry', 'ElementCollection', 'ExternalMap', 'Hash', 'Bundle', 'IndividualElement', 'Person', 'PositiveIntegerRange', 'Organization', 'ExternalIdentifier', 'Annotation', 'Agent', 'CreationInfo', 'LifecycleScopedRelationship', 'PackageVerificationCode', 'Relationship', 'Bom', 'Element'}



### 3. Finding all conditional classes

In [26]:
class_parents = {class_name: set() for class_name in classes}

for parent_name, definition in schema["$defs"].items():

    def search_for_ref(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                if key == "$ref" and isinstance(value, str):
                    referenced_class = value.split('/')[-1]
                    if referenced_class in classes:
                        class_parents[referenced_class].add(parent_name)
                else:
                    search_for_ref(value)
        elif isinstance(obj, list):
            for item in obj:
                search_for_ref(item)

    search_for_ref(definition)


### 4. Printing results

In [27]:
for class_name, parents in class_parents.items():
    print(f"Breadth for class '{class_name}': {len(parents)}, parents: {parents}")

Breadth for class 'NamespaceMap': 1, parents: {'SpdxDocument'}
Breadth for class 'Artifact': 3, parents: {'security_Vulnerability', 'software_SoftwareArtifact', 'ExternalMap'}
Breadth for class 'DictionaryEntry': 3, parents: {'dataset_DatasetPackage', 'ai_AIPackage', 'build_Build'}
Breadth for class 'ElementCollection': 2, parents: {'SpdxDocument', 'Bundle'}
Breadth for class 'ExternalMap': 1, parents: {'SpdxDocument'}
Breadth for class 'Hash': 1, parents: {'build_Build'}
Breadth for class 'Bundle': 1, parents: {'Bom'}
Breadth for class 'IndividualElement': 2, parents: {'NoneElement', 'NoAssertionElement'}
Breadth for class 'Person': 0, parents: set()
Breadth for class 'PositiveIntegerRange': 1, parents: {'software_Snippet'}
Breadth for class 'Organization': 1, parents: {'SpdxOrganization'}
Breadth for class 'ExternalIdentifier': 1, parents: {'Element'}
Breadth for class 'Annotation': 0, parents: set()
Breadth for class 'Agent': 6, parents: {'Person', 'Organization', 'Artifact', 'Softw

### 5. Saving results

In [28]:
import csv

with open(OUTPUT_FILE, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["class_name", "breadth", "parents"])
    for c, p in class_parents.items():
        w.writerow([c, len(p), "-".join(p)])

print(f"Saved class hierarchy info to {OUTPUT_FILE}")


Saved class hierarchy info to classes_breadth.csv
